# 企業共通データビュー

正本データは `report/company_analysis/data/*.yaml` です。  
この notebook は `tool/render_company_common_table.py` の集計ロジックを呼び出して、VSCode 上で並び替えや確認をしやすくするためのビューです。  
1つ目は公式データをそのまま表示し、2つ目は公式欠損時に非公式データで補完した表です。  
`初任` `学士初` `修士初` `博士初` `年収` は万円単位で表示しています。

In [1]:
from html import escape
from pathlib import Path
import sys

import pandas as pd
from IPython.display import HTML, display

ROOT = Path.cwd().resolve()
for cand in [ROOT, *ROOT.parents]:
    if (cand / 'tool').exists() and (cand / 'document').exists():
        ROOT = cand
        break
else:
    raise RuntimeError('repo root not found from current working directory')

sys.path.insert(0, str(ROOT / 'tool'))

from render_company_common_table import (
    HEADERS,
    collect_raw_rows,
    collect_raw_rows_with_unofficial_fallback,
)

paths = sorted((ROOT / 'report/company_analysis/data').rglob('*.yaml'))
data_root = ROOT / 'report/company_analysis/data'
report_links = {
    path: f"report/company_analysis/companies/{path.relative_to(data_root).with_suffix('.md').as_posix()}"
    for path in paths
}

def link_company_names(df):
    linked = df.copy()
    name_col = '会社' if '会社' in linked.columns else '会社名'
    linked[name_col] = [
        f'<a href="{report_links[path]}">{escape(name)}</a>'
        for path, name in zip(paths, linked[name_col], strict=True)
    ]
    return linked

def to_man_yen(value):
    if pd.isna(value):
        return None
    return round(float(value) / 10000, 1)

def build_display_df(rows):
    df = pd.DataFrame(rows, columns=HEADERS).drop(columns=['slug', '分析対象', '採用主体'])
    yen_columns = ['初任給', '学士初任給', '修士初任給', '博士初任給', '平均年収']
    for col in yen_columns:
        df[col] = df[col].map(to_man_yen)
    df = df.rename(columns={
        '会社名': '会社',
        '採用職種': '職種',
        '統合最終評価': '総合',
        '博士人材の評価': 'PhD',
        '仕事内容・配属確度': '職務',
        '研究開発・技術環境': 'R&D',
        '処遇・働き方': '処遇',
        '選考コストと評価の納得感': '選考',
        '企業基盤・安定性': '安定',
        '初任給': '初任',
        '学士初任給': '学士初',
        '修士初任給': '修士初',
        '博士初任給': '博士初',
        '平均年収': '年収',
        '月平均残業': '残業',
        '年間休日': '休日',
    })
    return link_company_names(df)

official_df = build_display_df(collect_raw_rows(paths))
fallback_df = build_display_df(collect_raw_rows_with_unofficial_fallback(paths))

try:
    from itables import init_notebook_mode, show
    init_notebook_mode(all_interactive=True)
    has_itables = True
except Exception:
    has_itables = False

def show_table(title, df):
    display(HTML(f'<h2>{escape(title)}</h2>'))
    if has_itables:
        show(df, allow_html=True)
    else:
        display(HTML(df.to_html(escape=False, index=False)))

show_table('公式データそのまま', official_df)
show_table('公式欠損を非公式で補完', fallback_df)

KeyError: "['分析対象', '採用主体'] not found in axis"

In [ ]:
# 例: CSV を再出力したいとき
# official_df.to_csv(ROOT / 'report/company_analysis/reviews/common_data_official_from_ipynb.csv', index=False)
# fallback_df.to_csv(ROOT / 'report/company_analysis/reviews/common_data_with_unofficial_fallback_from_ipynb.csv', index=False)